In [ ]:
import os
os.environ["HF_HOME"] = "/home/jovyan/.cache/huggingface"
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "0"
import torch
import polars as pl
from pathlib import Path
import time
import numpy as np
from evo2 import Evo2
import psutil # troubleshooting

# configuration
DATA_ROOT = Path.home() / "vambersky_t/Data"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"


LAYER = "blocks.28.mlp.l3"
SEQ_LEN = 10_000
BIN_SIZE = 50
N_BINS = SEQ_LEN // BIN_SIZE
EMB_DIM = 4096

TFS_TO_PROCESS = Path("tfs_to_process.txt").read_text().splitlines()
CHECKPOINT_EVERY = 500

assert DATA_ROOT.exists(),   f"Data root missing: {DATA_ROOT}"
assert WINDOWS_DIR.exists(), f"Windows dir missing: {WINDOWS_DIR}"
assert N_BINS == 200

print(f"Bins: {N_BINS} × {BIN_SIZE} bp = {SEQ_LEN} bp")
print(f"Per-peak tensor: ({N_BINS}, {EMB_DIM}) float16")
print(f"Checkpoint interval: every {CHECKPOINT_EVERY} peaks")

# load model
print(f"Loading model...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = Evo2("evo2_7b")
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Bins: 200 × 50 bp = 10000 bp
Per-peak tensor: (200, 4096) float16
Checkpoint interval: every 500 peaks
Loading model...


[05/05/26 14:52:07] INFO     httpx - INFO - HTTP Request: GET                                       ]8;id=13385652;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=13385653;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/arcinstitute/evo2_7b/revision/main                  
                             "HTTP/1.1 200 OK"                                                                     

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/home/jovyan/envs/evo2/lib/python3.12/site-packages/evo2/models.py:282: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(


...

                    INFO     StripedHyena - INFO - Parameter count for block 2: 205705216              ]8;id=13385712;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=13385713;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#672\672]8;;\

                    INFO     StripedHyena - INFO - Assigned layer_idx=3 to device='cuda:0'             ]8;id=13385718;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=13385719;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#671\671]8;;\

                    INFO     StripedHyena - INFO - Parameter count for block 3: 205533184              ]8;id=13385724;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=13385725;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#672\672]8;;\

 12%|█▎        | 4/32 [00:00<00:02, 11.33it/s]


RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1016, please report a bug to PyTorch. 

In [ ]:
# get the embeddings - mean of every 50, non overlapping (probrat s Vojtou - rozlišení, porovnatelnost)

def embed_sequence(sequence: str) -> np.ndarray:
    """
    Forward-pass one 10 kb sequence through Evo 2, extract embeddings
    from LAYER, bin into (200, 4096) via reshape + mean, return as
    float16 numpy array.

    Uses module-level `model` and `LAYER`.
    """
    input_ids = torch.tensor(
        model.tokenizer.tokenize(sequence),
        dtype=torch.int,
    ).unsqueeze(0).to("cuda:0")

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[LAYER],
        )

    emb = embeddings[LAYER][0]                 # (10000, 4096)

    # reshape + mean - as i did in the test, this works.
    binned = emb.reshape(N_BINS, BIN_SIZE, -1).mean(dim=1)

    return binned.cpu().to(torch.float16).numpy()

# test, delete
test_df = pl.read_csv(
    next((WINDOWS_DIR / "CTCF").glob("*.full_peaks.csv.gz")),
    n_rows=1,
)
test_emb = embed_sequence(test_df["sequence"][0])
print(test_emb.shape, test_emb.dtype) 

In [ ]:
# don't mess up my storage, pls

def get_output_paths(csv_path: Path) -> tuple[Path, Path, Path]:
    """
    Derive (final_npz, final_parquet, checkpoint_npz) from input CSV path.
    """
    base_name = csv_path.name.split(".full_peaks")[0]
    tf = base_name.split("__")[1]

    out_dir = EMBEDDINGS_DIR / tf
    out_dir.mkdir(parents=True, exist_ok=True)

    final_npz      = out_dir / f"{base_name}.embeddings.npz"
    final_parquet   = out_dir / f"{base_name}.peak_ids.parquet"
    checkpoint_npz  = out_dir / f"{base_name}.embeddings.checkpoint.npz"

    return final_npz, final_parquet, checkpoint_npz


In [ ]:
def process_csv_file(csv_path: Path) -> None:
    """
    Full embedding extraction for one CSV file of peak sequences.
    """
    final_npz, final_parquet, checkpoint_npz = get_output_paths(csv_path)

    if final_npz.exists() and final_parquet.exists():  # skip processed
        print(f"  SKIP (complete): {final_npz.name}")
        return

    df = pl.read_csv(csv_path)
    n_peaks = len(df)
    sequences = df["sequence"].to_list()
    peak_ids = df["peak_id"].to_list()

    bad = [i for i, s in enumerate(sequences) if len(s) != SEQ_LEN]
    if bad:
        print(f"  WARNING: {len(bad)} sequences != {SEQ_LEN} bp "
              f"(indices {bad[:5]}{'...' if len(bad) > 5 else ''}). "
              f"These will be skipped.")

    LOCAL_WORK_DIR = Path("/home/jovyan/embedding_work")
    LOCAL_WORK_DIR.mkdir(exist_ok=True)
    work_path     = LOCAL_WORK_DIR / f"{final_npz.stem}.work.bin"
    progress_path = LOCAL_WORK_DIR / f"{final_npz.stem}.progress.txt"

    bytes_per_emb = N_BINS * EMB_DIM * 2  # fp16 = 2 bytes per element

    # --- resume or fresh start ---
    if work_path.exists() and progress_path.exists():
        progress = progress_path.read_text().strip().split("\n")
        valid_peak_ids = progress[:-1]
        start_idx      = int(progress[-1])
        n_written      = len(valid_peak_ids)
        print(f"  RESUME: {start_idx}/{n_peaks} peaks scanned, "
              f"{n_written} valid embeddings on disk")
        work_fh = open(work_path, "r+b")
        work_fh.seek(n_written * bytes_per_emb)
    else:
        valid_peak_ids = []
        start_idx      = 0
        n_written      = 0
        work_fh = open(work_path, "wb")

    # --- inference loop ---
    t0 = time.time()
    try:
        for i in range(start_idx, n_peaks):
            seq = sequences[i]
            if len(seq) != SEQ_LEN:
                continue

            emb = embed_sequence(seq)
            work_fh.write(emb.tobytes())
            n_written += 1
            valid_peak_ids.append(peak_ids[i])

            if (i + 1) % 100 == 0:
                elapsed   = time.time() - t0
                done      = i + 1 - start_idx
                rate      = done / elapsed
                remaining = (n_peaks - i - 1) / rate if rate > 0 else float("inf")
                ram_gb    = psutil.Process().memory_info().rss / 1e9
                print(f"  [{i+1}/{n_peaks}] {rate:.1f} peaks/s  "
                      f"~{remaining / 60:.0f} min left  "
                      f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB  "
                      f"RAM {ram_gb:.1f} GB")

            if (i + 1) % CHECKPOINT_EVERY == 0:
                ckpt_t0 = time.time()
                work_fh.flush()
                os.fsync(work_fh.fileno())
                os.posix_fadvise(work_fh.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                progress_path.write_text(
                    "\n".join(valid_peak_ids) + "\n" + str(i + 1)
                )
                print(f"  CHECKPOINT at peak {i+1} ({time.time() - ckpt_t0:.1f}s)")

        # --- finalize ---
        work_fh.flush()
        os.fsync(work_fh.fileno())
        work_fh.close()

        sidecar = (
            pl.DataFrame({"peak_id": valid_peak_ids})
            .join(
                df.select(["peak_id", "chr", "start", "end", "center"]),
                on="peak_id",
                how="left",
            )
        )
        sidecar.write_parquet(final_parquet)
        
        save_t0 = time.time()
        trimmed = np.fromfile(work_path, dtype=np.float16).reshape(
            n_written, N_BINS, EMB_DIM
        )
        np.savez_compressed(final_npz, embeddings=trimmed)
        del trimmed
        print(f"  Final save took {time.time() - save_t0:.1f}s")

        elapsed_total = time.time() - t0
        print(f"  DONE: {n_written} peaks → {final_npz.name}  "
              f"{elapsed_total / 60:.1f} min")

        work_path.unlink()
        progress_path.unlink(missing_ok=True)
        print(f"  Working files removed")

    except BaseException:
        # Make sure the file handle is closed even on KeyboardInterrupt / OOM
        try:
            work_fh.close()
        except Exception:
            pass
        raise

In [ ]:
csv_files = []
for entry in TFS_TO_PROCESS:
    tf = entry.split("__")[1]
    path = WINDOWS_DIR / tf / f"{entry}.full_peaks.csv.gz"
    if not path.exists():
        print(f"WARNING: Not found: {path.name}")
        continue
    csv_files.append(path)

print(f"\nTotal: {len(csv_files)} files")
for f in csv_files:
    print(f"  {f.parent.name}/{f.name}")

In [ ]:
# npz, parquet, _ = get_output_paths(test_csv)
# data = np.load(npz)
# ids = pl.read_parquet(parquet)
# print(data["embeddings"].shape, data["embeddings"].dtype)
# print(ids.shape)
# print(ids.head())


In [ ]:
# files_to_run = [
#     WINDOWS_DIR / "MYC"  / "ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz",
#     WINDOWS_DIR / "MYC"  / "ENCFF700CXD__MYC__H1.full_peaks.csv.gz",
#     WINDOWS_DIR / "CTCF" / "ENCFF017XLW__CTCF__GM12878.full_peaks.csv.gz",
#     WINDOWS_DIR / "RANDOM" / "RANDOM__RANDOM__hg19.full_peaks.csv.gz",
# ]

# for i, csv_path in enumerate(files_to_run):
#     print(f"\n[{i+1}/{len(files_to_run)}] {csv_path.parent.name}/{csv_path.name}")
#     process_csv_file(csv_path)
#     torch.cuda.empty_cache()

In [ ]:
pipeline_t0 = time.time()
for idx, csv_path in enumerate(csv_files):
    print(f"\n[{idx+1}/{len(csv_files)}] {csv_path.parent.name}/{csv_path.name}")
    process_csv_file(csv_path)
    torch.cuda.empty_cache()

elapsed = time.time() - pipeline_t0
print(f"\n{'='*50}")
print(f"Pipeline complete: {elapsed / 3600:.1f} hours")
